In [3]:
import pandas as pd
import numpy as np
#导入时间包
import pytz


In [ ]:
df=pd.read_csv('data/2019.csv')
#时间数据要注意时区是否一致
#这里是测试数据只有第一天的数据在这里进行的时间修改记得加在后面
#将文件去除时区方便导入mysql,或者可以一开始就将删掉原来的时区
utc=pytz.UTC
start_time=pd.to_datetime('2019-11-01 00:00:00').tz_localize(utc)
end_time=pd.to_datetime('2019-11-30 23:59:59').tz_localize(utc)
df['event_time']=pd.to_datetime(df['event_time'])
df=df[(df['event_time']>=start_time)&(df['event_time']<=end_time)]
#去除多余字段
df=df.drop('user_session',axis=1)
#步骤1:删除缺失值(核心字段缺失直接删,非核心字段缺失用平均数或者用中位数,附近的数据来填充)
nan=len(df)
df=df.dropna(subset=['user_id','product_id','event_time','event_type'])
print(f'删除缺失值{nan-len(df)},剩余{len(df)}条')
#填充缺失值,按它的对应类别来进行取值填充,如果该类别下全部为空值就填充空值,再处理一次dropna删掉全部空值
df['category_id']=df.groupby('product_id')['category_id'].transform(lambda x:x.fillna(x.mode()[0] if not x.mode().empty else np.nan))
df['price']=df.groupby('product_id')['price'].transform(lambda x:x.fillna(x.mean() if not x.mean() else np.nan))
df['brand']=df.groupby('category_id')['brand'].transform(lambda x:x.fillna(x.mode()[0] if not x.mode().empty else np.nan))
df['category_code']=df.groupby('category_id')['category_code'].transform(lambda x:x.fillna(x.mode()[0] if not x.mode().empty else np.nan))
df=df.dropna()

#步骤2:去重:去除重复的记录
dcs=len(df)
df=df.drop_duplicates()
print(f'去除重复值{dcs-len(df)},剩余{len(df)}条')
#步骤3:处理时间数据
#1,将时间数据类型改为datetime方便后续操作
# df['event_time']=pd.to_datetime(df['event_time'])
#2,提取衍生时间数据字段
df['date']=df['event_time'].dt.date
df['hour']=df['event_time'].dt.hour
df['weekday']=df['event_time'].dt.weekday
df['weekend']=df['weekday'].apply(lambda x:1 if x>=5 else 0)
#过滤异常值
type=['view', 'cart', 'purchase']
df=df[df['event_type'].isin(type)]
df=df[df['price']>0]
# df['event_time']=df[(df['event_time']>=pd.to_datetime(''))&&(df['event_time']<=pd.to_datetime(''))]
#步骤5:字段类型统一
df['product_id']=df['product_id'].astype(str)
df['category_id']=df['category_id'].astype(str)
df['user_id']=df['user_id'].astype(str)
#去除时区
df['event_time']=df['event_time'].dt.tz_localize(None)
#保存清洗后的数据
df.to_csv('data/clean_2019.csv',index=False,encoding='utf-8')

删除缺失值0,剩余4635837条
